In [1]:
import numpy as np
from pyscf import gto, sftda
from pyscf.sftda.uhf_sf import get_ab_sf
from pyscf.sftda.tools_td import transition_analyze, spin_square
import sys
sys.path.append('/home/peter/Research/ZeroExcitationEnergy/code/src')
sys.path.append('/public1/home/scg0213/Jobs_WT/ZeroExcitationEnergy/src')
from tools import extract_state

In [2]:
r_list = np.concatenate((np.linspace(0.4, 3.0, 53), np.linspace(3.2, 6.0, 15)))
fun_list = ['SVWN', 'BLYP', 'B3LYP']
bas = '6-31G'

In [3]:
e_list = []
for r in r_list:
    mol = gto.M(atom=f'H; H 1 {r}', basis=bas, charge=0, spin=0)
    mf = mol.RKS(xc='B3LYP')
    mf.kernel()
    e_list.append(mf.e_tot)
    if not mf.converged:
        break

/home/peter/Research/ConicalIntersection/PySCF_HPC/pyscf/dft/libxc.py:1102: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -0.976178685946584
converged SCF energy = -1.05218636130629
converged SCF energy = -1.10261773345734
converged SCF energy = -1.135552494749
converged SCF energy = -1.15626548499005
converged SCF energy = -1.16830592563674
converged SCF energy = -1.17412356600679
converged SCF energy = -1.1754478347314
converged SCF energy = -1.17352304930133
converged SCF energy = -1.16925744634764
converged SCF energy = -1.16332050537059
converged SCF energy = -1.15620872823994
converged SCF energy = -1.14829183430875
converged SCF energy = -1.1398460270586
converged SCF energy = -1.13107829613409
converged SCF energy = -1.12214420540119
converged SCF energy = -1.11316089027756
converged SCF energy = -1.1042165336939
converged SCF energy = -1.09537734697172
converged SCF energy = -1.08669272843673
converged SCF energy = -1.0781991880069
converged SCF energy = -1.06992332218226
converged SCF energy = -1.06188412812492
converged SCF energy = -1.05409477361021
converged SCF energy 

In [4]:
np.save('RKS', np.array(e_list))

In [3]:
for fun in fun_list:
    print(f'Calculating {fun}...')
    s2_list = []
    td_list = []
    for r in r_list:
        mol = gto.M(atom=f'H; H 1 {r}', basis=bas, charge=0, spin=2)
        mf = mol.UKS(xc=fun)
        mf.kernel()

        a, b = get_ab_sf(mf, collinear_samples=50)
        A_baba, A_abab = a
        B_baab, B_abba = b

        n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
        n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

        A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
        B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
        B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
        A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

        Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                                [-B_baab_2d,-A_baba_2d]])

        eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

        idxt = eigenvals.real.argsort()
        eigenvals = eigenvals[idxt]
        eigenvecs = eigenvecs[:, idxt]

        norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
        norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

        sfd_eigenvals = eigenvals[norms>0]
        sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

        def norm_xy(z):
            x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
            y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
            norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
            norm = np.sqrt(1./norm)
            return ((0,x*norm), (y*norm,0))

        mftd1 = sftda.TDA_SF(mf)
        mftd1.nstates = len(sfd_eigenvals)
        mftd1.extype = 1
        mftd1.collinear_samples = 50
        mftd1.e = sfd_eigenvals
        mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

        target = extract_state(mf, mftd1, tdtype='TDDFT', Smin=0.0, Smax=0.9)[0]

        print(transition_analyze(mf, mftd1, mftd1.e[target], mftd1.xy[target], tdtype='TDDFT'))
        td_list.append(mftd1.e[target] + mf.e_tot)
        s2_list.append(spin_square(mf, mftd1.xy[target], extype=1))
    np.save('r.npy', r_list)
    np.save(f'{fun}_e.npy', td_list)
    np.save(f'{fun}_s2.npy', s2_list)
    print('-----------------------')

Calculating SVWN...
converged SCF energy = -0.316033582323343  <S^2> = 2  2S+1 = 3
50
1-th state with excitation energy = -0.5303, S^2 = 0.023 Norms: 1.00@(2, 1), 0.00@(1, 2)
2-th state with excitation energy = 0.0918, S^2 = 0.012 Norms: 0.53@(1, 1), 0.45@(2, 2)
3-th state with excitation energy = 0.9226, S^2 = 0.000 Norms: 1.00@(1, 2), 0.00@(1, 4)
Excited energy : -14.430942063555257 eV.
<S2> value : 0.022987459722534798.
The main and second norm:
0.9979035281538503 0.0009526820609700232
The main and second norm to orbital pair:
(2.0, 1.0) (1.0, 2.0)


None
converged SCF energy = -0.425669221813001  <S^2> = 2  2S+1 = 3
50
1-th state with excitation energy = -0.5001, S^2 = 0.022 Norms: 1.00@(2, 1), 0.00@(1, 2)
2-th state with excitation energy = 0.0989, S^2 = 0.012 Norms: 0.53@(1, 1), 0.46@(2, 2)
3-th state with excitation energy = 0.8764, S^2 = 0.000 Norms: 1.00@(1, 2), 0.00@(1, 4)
Excited energy : -13.608163277834208 eV.
<S2> value : 0.022307789871840455.
The main and second norm:
0.

/home/peter/Research/ConicalIntersection/PySCF_HPC/pyscf/dft/libxc.py:1102: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -0.35603522075995  <S^2> = 2  2S+1 = 3
50
1-th state with excitation energy = -0.5381, S^2 = 0.035 Norms: 0.99@(2, 1), 0.00@(1, 4)
2-th state with excitation energy = 0.0989, S^2 = 0.283 Norms: 0.82@(1, 1), 0.16@(2, 2)
3-th state with excitation energy = 0.9643, S^2 = 0.004 Norms: 0.98@(1, 2), 0.01@(1, 4)
Excited energy : -14.641461031692938 eV.
<S2> value : 0.034615234785755344.
The main and second norm:
0.9943383350144369 0.003358022868793289
The main and second norm to orbital pair:
(2.0, 1.0) (1.0, 4.0)


None
converged SCF energy = -0.46484725769755  <S^2> = 2  2S+1 = 3
50
1-th state with excitation energy = -0.5091, S^2 = 0.032 Norms: 0.99@(2, 1), 0.00@(1, 4)
2-th state with excitation energy = 0.1044, S^2 = 0.204 Norms: 0.77@(1, 1), 0.20@(2, 2)
3-th state with excitation energy = 0.9151, S^2 = 0.003 Norms: 0.98@(1, 2), 0.02@(1, 4)
Excited energy : -13.852244295380794 eV.
<S2> value : 0.0324784405130778.
The main and second norm:
0.9931329254745419 0.003904

In [4]:
np.load('B3LYP_e.npy')

array([ 0.10604227, -0.90301904, -1.09391951, -1.12432116, -1.11060487,
       -1.08402143, -1.05687772, -1.03246265, -1.01155707, -0.99546084,
       -0.98351387, -0.97514232, -0.96948692, -0.96477283, -0.9614907 ,
       -0.95983692, -0.95893681, -0.95754892, -0.95700337, -0.95657411,
       -0.95639319, -0.95609643, -0.95602357, -0.95586541, -0.95595109,
       -0.95576676, -0.95572471, -0.95569956, -0.95569679, -0.9556843 ,
       -0.95567441, -0.95567539, -0.95566529, -0.95567126, -0.95566312,
       -0.95566792, -0.95567076, -0.95567492, -0.95567306, -0.9556714 ])

In [8]:
np.load('SVWN_e.npy')

array([ 0.136001  , -0.86737228, -1.05186332, -1.08111163, -1.07057901,
       -1.04969312, -1.0275744 , -1.00758548, -0.99095585, -0.97793809,
       -0.9682849 , -0.96147316, -0.95686759, -0.95385792, -0.9519403 ,
       -0.9507397 , -0.94999592, -0.94953721, -0.94925389, -0.94907819,
       -0.94896826, -0.94889959, -0.94885607, -0.94882867, -0.94881201,
       -0.94880188, -0.94879484, -0.94878997, -0.9487885 , -0.94878958,
       -0.94878957, -0.94878653, -0.94878234, -0.94878031, -0.94878212,
       -0.9487867 , -0.94879126, -0.94879347, -0.94879301, -0.94879073])